# Industry Deployment: Status, Evidence and Operations

**Module 07 · Runtime Assurance and Operations** · Notebook 3 of 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/07_Runtime_Assurance_and_Operations/03_industry_deployment.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~50 min | [ODD and runtime monitoring](01_odd_and_runtime_monitoring.ipynb), [Statistical safety validation](../06_AI_Safety_ISO_PAS_8800/07_statistical_safety_validation.ipynb) | NHTSA SGO 2021-01 (3rd amendment), UN R157, UL 4600 (3rd ed., 2023) |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

After this notebook you will be able to:

- describe the state of automated-driving deployment as of October 2026, with dated and sourced facts;
- distinguish the deployment models robotaxi, driverless trucking, consumer Level 3 and supervised Level 2, and who carries the fallback and the liability in each;
- interpret published safety-impact data (Waymo) statistically, and explain why injury outcomes can be demonstrated long before fatality outcomes;
- separate remote assistance from remote driving and treat remote-operations staffing as a safety parameter;
- explain what the NHTSA Standing General Order requires and what crash-report data can and cannot show.

> **How to read the numbers.** Every time-sensitive fact below carries an "as of" date and a source. The field changes monthly; check the sources before reusing a number.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
OI = {"blue": "#0072B2", "orange": "#E69F00", "green": "#009E73", "red": "#D55E00",
      "purple": "#CC79A7", "sky": "#56B4E9", "yellow": "#F0E442", "black": "#000000"}
pd.set_option("display.max_colwidth", 110)
AS_OF = "October 2026"
print("numpy", np.__version__, "| pandas", pd.__version__, "| facts as of", AS_OF)

---
## 1. From promises to deployment

Around 2015–2018 many companies announced driverless cars "by 2020". The following years showed that validation, operations and public trust were much harder than the core driving demo. By 2026 the picture is no longer "no deployment anywhere": driverless robotaxis operate commercially in limited but growing ODDs in the US and China, driverless trucks run on defined highway lanes, and L3 systems are approved for consumer cars in Germany and piloted in China — while general-purpose, go-anywhere automation (Level 5) does not exist.

In [ ]:
# Dated milestones; every row carries its source (see the References for full links).
milestones = pd.DataFrame([
    ("2018-03-18", "Uber ATG test vehicle kills a pedestrian in Tempe, AZ (safety driver on board)", "Incident", "NTSB HAR-19/03"),
    ("2023-10-24", "California DMV suspends Cruise's driverless permits after the 2 Oct 2023 pedestrian-dragging crash", "Incident", "CA DMV; electrive 2024"),
    ("2024-12-10", "GM announces it will stop funding Cruise's robotaxi business", "Industry", "Fortune / AP, 10 Dec 2024"),
    ("2024-12-17", "KBA extends Mercedes-Benz DRIVE PILOT (L3) approval to 95 km/h on German motorways", "Approval", "heise / electrive, Dec 2024"),
    ("2025-04-24", "NHTSA issues third amended Standing General Order (effective 16 Jun 2025)", "Regulation", "NHTSA; Sidley, Apr 2025"),
    ("2025-04-15", "Aurora begins driverless trucking, Dallas-Houston (April 2025)", "Deployment", "Aurora IR, Feb 2026 (\"April 2025\")"),
    ("2025-09-10", "Zoox opens its purpose-built robotaxi service to the public in Las Vegas", "Deployment", "TechCrunch, 10 Sep 2025"),
    ("2025-11-12", "Waymo starts freeway rides in San Francisco, Los Angeles and Phoenix", "Deployment", "TechCrunch, 12 Nov 2025"),
    ("2025-12-15", "China's MIIT grants first conditional L3 product approvals (Changan, BAIC Arcfox)", "Approval", "TechNode, 15 Dec 2025"),
    ("2026-01-22", "Tesla begins some Austin Robotaxi rides without an in-car safety monitor (chase cars reported)", "Deployment", "Electrek, 22 Jan 2026"),
    ("2026-06-24", "WP.29 adopts a UN Regulation and a GTR for automated driving systems (June 2026)", "Regulation", "UNECE (via CCAM.eu / Bennett Jones)"),
    ("2026-09-01", "Waymo opens Denver, San Diego and Tampa: fully driverless rides in 14 cities", "Deployment", "Waymo blog, 1 Sep 2026"),
], columns=["date", "event", "category", "source"])
milestones["date"] = pd.to_datetime(milestones["date"])
cat_colour = {"Incident": OI["red"], "Industry": OI["purple"], "Approval": OI["orange"],
              "Regulation": OI["blue"], "Deployment": OI["green"]}
fig, ax = plt.subplots(figsize=(12, 5.5))
for i, r in milestones.iterrows():
    ax.scatter(r["date"], i, color=cat_colour[r["category"]], s=60, zorder=3)
    ax.text(r["date"] + pd.Timedelta(days=25), i, r["event"], va="center", fontsize=8)
for c, col in cat_colour.items():
    ax.scatter([], [], color=col, label=c)
ax.set_yticks([]); ax.invert_yaxis(); ax.set_xlabel("date")
ax.set_xlim(pd.Timestamp("2017-06-01"), pd.Timestamp("2031-06-01"))
ax.set_title(f"Selected AV deployment milestones (sourced; as of {AS_OF})")
ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()
milestones.assign(date=milestones["date"].dt.date)

---
## 2. Deployment models

The deployment model determines who drives, who performs the fallback and — usually — who is liable.

In [ ]:
models = pd.DataFrame([
    ["Robotaxi (L4)", "ADS within a geofenced ODD; no driver", "ADS -> MRC; remote assistance may advise",
     "Operator (fleet owner)", "Waymo, Baidu Apollo Go, Zoox; Tesla Robotaxi (in transition)"],
    ["Driverless trucking (L4)", "ADS on defined hub-to-hub highway lanes", "ADS -> MRC (shoulder stop); terminal and remote support",
     "Operator / carrier", "Aurora (Texas, New Mexico, Arizona)"],
    ["Consumer L3 ('eyes off')", "ADS within a narrow ODD; driver is the fallback-ready user",
     "Transition demand -> driver; else MRM (UN R157)", "Manufacturer while the system is engaged (jurisdiction-dependent)",
     "Mercedes DRIVE PILOT, BMW Personal Pilot L3; China L3 pilots"],
    ["Supervised L2 ('hands off' possible)", "Driver supervises continuously; no ADS", "Driver at all times",
     "Driver", "Tesla FSD (Supervised), GM Super Cruise and others"],
], columns=["Deployment model", "Who drives", "Fallback", "Typical liability locus", "Examples"])
models

Notes:

- **Tesla FSD (Supervised)** is a Level 2 driver-assistance feature: the driver is responsible at all times. The Tesla *Robotaxi* service is a separate operation whose supervision arrangements have changed over time (in-car monitor, then chase cars reported in January 2026); no audited safety data were published as of the sources used here.
- **Level 3** is defined by the *fallback-ready user*; see [notebook 01](01_odd_and_runtime_monitoring.ipynb) for the UN R157 transition-demand sequence. In Germany, DRIVE PILOT operates up to 95 km/h in the right lane behind a lead vehicle (KBA approval, December 2024); UN R157 permits more than 60 km/h only for systems capable of an MRM lane change.
- Regulation and type approval (UN R157, the June 2026 UN ADS Regulation, EU 2022/1426, Germany's Act on Autonomous Driving) are covered in [Module 08](../08_Standards_Regulation_and_Safety_Case/02_regulation_and_type_approval.ipynb).

---
## 3. Landscape (as of October 2026)

Selected operators and products. "Published scale / evidence" lists only figures the companies themselves (or their filings) published; where nothing was published, the table says so.

In [ ]:
landscape = pd.DataFrame([
    ["Waymo (US)", "L4 robotaxi", "Fully driverless public rides in 14 cities; freeway rides in SF, LA, Phoenix since Nov 2025",
     ">500k paid rides/week (Alphabet Q1 2026 call); 271.3M rider-only miles through Jun 2026", "Sep 2026",
     "waymo.com/blog/2026/09; waymo.com/safety/impact"],
    ["Baidu Apollo Go (CN + intl.)", "L4 robotaxi", "28 cities worldwide, incl. fully driverless operations in Dubai",
     "3.2M fully driverless rides in Q1 2026; >22M cumulative (Apr 2026); >240M fully driverless km", "Aug 2026",
     "Baidu Q1 and Q2 2026 results (SEC 6-K)"],
    ["Zoox (US)", "L4 robotaxi, purpose-built vehicle", "Public service in Las Vegas (since Sep 2025); San Francisco early-rider programme",
     "No ride volumes published", "Nov 2025", "TechCrunch 10 Sep 2025 and 18 Nov 2025"],
    ["Tesla Robotaxi (US)", "Driverless service in ramp-up", "Austin; some rides without in-car monitor since Jan 2026, with chase cars reported",
     "No audited safety data published", "Jan 2026", "Electrek 22 Jan 2026"],
    ["Aurora (US)", "L4 trucking", "Driverless lanes incl. Dallas-Houston and ~1,000-mile Fort Worth-Phoenix",
     "250k driverless miles, no collisions attributed to the Aurora Driver (Jan 2026); target >200 trucks by end 2026", "Feb 2026",
     "Aurora IR 11 Feb 2026"],
    ["Mercedes-Benz DRIVE PILOT", "L3 consumer", "German motorways up to 95 km/h in the right lane behind a lead vehicle",
     "Approval only; no usage data", "Dec 2024", "heise / electrive Dec 2024"],
    ["BMW Personal Pilot L3", "L3 consumer", "7 Series, Germany, motorways up to 60 km/h", "Approval only", "Aug 2024",
     "BMW Group press release"],
    ["Changan Deepal SL03 / BAIC Arcfox Alpha S", "L3 pilot", "Designated expressways in Chongqing / Beijing (reported max. 50 / 80 km/h)",
     "Pilot under supervision", "Dec 2025", "TechNode 15 Dec 2025"],
    ["Cruise (GM)", "L4 robotaxi - discontinued", "Permits suspended Oct 2023; GM ended robotaxi funding Dec 2024",
     "-", "Dec 2024", "Fortune / AP 10 Dec 2024"],
], columns=["Company / product", "Model", "Status", "Published scale / evidence", "As of", "Source"])
landscape

Sources: [Waymo, 1 Sep 2026](https://waymo.com/blog/2026/09/ride-in-denver-san-diego-tampa/); [Waymo Safety Impact](https://waymo.com/safety/impact/); [TechCrunch, Waymo freeways, 12 Nov 2025](https://techcrunch.com/2025/11/12/waymo-robotaxis-are-now-giving-rides-on-freeways-in-these-3-cities/); weekly rides as stated on Alphabet's Q1 2026 call (reported e.g. by [InsideEVs](https://insideevs.com/news/791245/waymo-rides-per-week-2026-doubled/)); Baidu [Q1 2026](https://www.sec.gov/Archives/edgar/data/0001329099/000119312526228123/d156481dex991.htm) and [Q2 2026](https://www.sec.gov/Archives/edgar/data/0001329099/000119312526355431/d112854dex991.htm) results; [TechCrunch, Zoox, 10 Sep 2025](https://techcrunch.com/2025/09/10/zoox-opens-its-las-vegas-robotaxi-service-to-the-public/) and [18 Nov 2025](https://techcrunch.com/2025/11/18/zoox-to-begin-offering-robotaxi-rides-to-public-in-san-francisco/); [Electrek, Tesla, 22 Jan 2026](https://electrek.co/2026/01/22/tesla-didnt-remove-the-robotaxi-safety-monitor-it-just-moved-them-to-a-trailing-car/); [Aurora, 11 Feb 2026](https://ir.aurora.tech/news-events/press-releases/detail/132/aurora-triples-driverless-network-to-10-routes-and-prepares-to-expand-across-u-s-sun-belt); [electrive, Mercedes, 17 Dec 2024](https://www.electrive.com/2024/12/17/mercedes-receives-approval-for-new-autonomous-system-in-germany/); [BMW Group press release](https://press.bmwgroup.com/asia/article/detail/T0443363EN/road-to-autonomous-driving-bmw-is-the-first-car-manufacturer-to-receive-approval-for-the-combination); [TechNode, 15 Dec 2025](https://technode.com/2025/12/15/china-grants-market-access-to-its-first-batch-of-l3-autonomous-driving-vehicles/); [Fortune, 10 Dec 2024](https://fortune.com/2024/12/10/general-motors-stop-funding-cruise-robotaxi-business).

Baidu's Q2 2026 release notes that ride volume was affected by "operational adjustments" in some domestic cities; the release does not state a quarterly ride count, so none is given here.

---
## 4. Safety evidence: what the data can and cannot show

### 4.1 Published safety-impact data (Waymo)

Waymo's Safety Impact hub compares its rider-only (RO) crash rates with human benchmarks for the same cities and road types. As of the data through **June 2026** (271.3 million RO miles in Phoenix, the San Francisco Bay Area, Los Angeles, Austin and Atlanta), the hub reports, for all locations combined: 95 % fewer serious-injury-or-worse crashes, 82 % fewer airbag-deployment crashes, 82 % fewer injury-reported crashes, 93 % fewer pedestrian-injury, 86 % fewer cyclist-injury and 82 % fewer motorcyclist-injury crashes. The methodology is described in peer-reviewed papers, e.g. Kusano et al. (2024) for the first 7.1 million miles.

The cell below reproduces these figures and back-calculates approximate event counts from the rounded percentages and the published "fewer crashes" numbers. **These counts are approximations**, not Waymo's data.

In [ ]:
# Waymo Safety Impact hub, data through June 2026 (271.3M rider-only miles; Phoenix, SF Bay Area,
# Los Angeles, Austin, Atlanta). Reductions vs. a human benchmark over the same distance, all locations.
hub = pd.DataFrame([
    ("Serious injury or worse", 95, 55),
    ("Airbag deployment (any vehicle)", 82, 357),
    ("Any injury reported", 82, 841),
    ("Pedestrian injury", 93, 87),
    ("Cyclist injury", 86, 56),
    ("Motorcyclist injury", 82, 36),
], columns=["outcome", "reduction [%]", "fewer crashes"])
RO_MILES = 271.3e6

# Back-calculate approximate counts from the rounded published figures:
#   benchmark - waymo = fewer,  waymo / benchmark = 1 - reduction
hub["benchmark (approx.)"] = hub["fewer crashes"] / (hub["reduction [%]"] / 100)
hub["Waymo (approx.)"] = hub["benchmark (approx.)"] - hub["fewer crashes"]
hub["Waymo per M miles"] = hub["Waymo (approx.)"] / (RO_MILES / 1e6)
hub["benchmark per M miles"] = hub["benchmark (approx.)"] / (RO_MILES / 1e6)
display(hub.round(3))

fig, ax = plt.subplots(figsize=(10, 4))
y = np.arange(len(hub))
ax.barh(y - 0.2, hub["benchmark per M miles"], height=0.4, color=OI["orange"], label="human benchmark (approx.)")
ax.barh(y + 0.2, hub["Waymo per M miles"], height=0.4, color=OI["blue"], label="Waymo (approx.)")
ax.set_yticks(y, hub["outcome"]); ax.invert_yaxis(); ax.set_xscale("log")
ax.set_xlabel("crashes per million rider-only miles (log scale)")
ax.set_title("Waymo vs. human benchmark, 271.3M miles through Jun 2026\n(rates back-calculated from rounded published figures)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### 4.2 How certain are these reductions?

Treating the back-calculated counts as Poisson observations with equal exposure gives an exact confidence interval for the rate ratio:

In [ ]:
def rate_ratio_ci(k_a, k_b, conf=0.95):
    """Exact CI for the ratio of two Poisson rates with equal exposure (conditional binomial method)."""
    n = k_a + k_b
    a = 1 - conf
    lo = stats.beta.ppf(a / 2, k_a, k_b + 1) if k_a > 0 else 0.0
    hi = stats.beta.ppf(1 - a / 2, k_a + 1, k_b)
    return lo / (1 - lo), hi / (1 - hi)

rows = []
for _, r in hub.iterrows():
    k_w, k_h = int(round(r["Waymo (approx.)"])), int(round(r["benchmark (approx.)"]))
    lo, hi = rate_ratio_ci(k_w, k_h)
    rows.append({"outcome": r["outcome"], "Waymo count ~": k_w, "benchmark ~": k_h,
                 "rate ratio": round(k_w / k_h, 3), "95% CI": f"[{lo:.3f}, {hi:.3f}]",
                 "reduction lower bound [%]": round(100 * (1 - hi), 1)})
ci = pd.DataFrame(rows)
display(ci)
print("Text summary (derived from the table):")
for _, r in ci.iterrows():
    print(f"- {r['outcome']}: about {r['Waymo count ~']} vs {r['benchmark ~']} expected; "
          f"reduction at least {r['reduction lower bound [%]']} % with 95 % confidence (treating the benchmark as a count).")
print("Caveat: the benchmark itself is an estimate with its own uncertainty and adjustments (e.g. under-reporting of")
print("human crashes); Waymo's published confidence intervals account for this differently, so these numbers only")
print("illustrate the statistics and do not replace the peer-reviewed analyses.")

### 4.3 Why fatalities are different

Fatalities are much rarer than injury crashes. Using the US human rate of 1.09 fatalities per 100 million miles (2013), as in Kalra & Paddock (2016), we compute the exposure needed for different claims. The zero-failure formula is the same as `avsafety.validation.miles_to_demonstrate` (see repo `avsafety/`).

In [ ]:
# How many miles would be needed to show a lower *fatality* rate than humans?
# Reference rate: 1.09 fatalities per 100 million miles (US, 2013), the value used by Kalra & Paddock (2016).
R_H = 1.09e-8

def zero_failure_miles(rate, conf=0.95):
    return -np.log(1 - conf) / rate          # = avsafety.validation.miles_to_demonstrate(rate, conf, 0)

def miles_for_power(reduction, rate_h=R_H, alpha=0.05, power=0.8):
    """Miles at which a one-sided exact Poisson test of H0: rate >= rate_h rejects with the given power,
    if the true AV rate is (1 - reduction) * rate_h."""
    rate_a = (1 - reduction) * rate_h
    for n in np.logspace(7, 12, 400):
        lam0, lam1 = rate_h * n, rate_a * n
        k_crit = stats.poisson.ppf(alpha, lam0)            # reject if observed <= k_crit (approximately level alpha)
        while stats.poisson.cdf(k_crit, lam0) > alpha and k_crit >= 0:
            k_crit -= 1
        if k_crit >= 0 and stats.poisson.cdf(k_crit, lam1) >= power:
            return n
    return np.nan

need = pd.DataFrame([{"claim": "rate below human rate, zero fatalities observed (95 %)",
                      "miles needed [M]": zero_failure_miles(R_H) / 1e6}] +
                    [{"claim": f"true reduction {int(100 * x)} %, 80 % power, alpha 0.05",
                      "miles needed [M]": miles_for_power(x) / 1e6} for x in (0.2, 0.5, 0.9)])
need["x Waymo RO miles (Jun 2026)"] = need["miles needed [M]"] * 1e6 / RO_MILES
display(need.round(1))
m = need.set_index("claim")["miles needed [M]"]
x = need.set_index("claim")["x Waymo RO miles (Jun 2026)"]
k0, k20, k90 = need["claim"].iloc[0], need["claim"].iloc[1], need["claim"].iloc[3]
print(f"A zero-fatality demonstration needs ~{m[k0]:.0f}M miles (for scale: Waymo's rider-only total was "
      f"{RO_MILES / 1e6:.0f}M in Jun 2026).")
print(f"Showing a 20 % lower fatality rate with 80 % power needs ~{m[k20] / 1e3:.1f} billion miles "
      f"({x[k20]:.0f}x the Jun 2026 total); if the true reduction were 90 %, ~{m[k90]:.0f}M miles would suffice.")
print("Kalra & Paddock (2016) reach the same order of magnitude (billions of miles) with a slightly different test.")
ratios = hub.set_index("outcome")["benchmark per M miles"] / (R_H * 1e6)
print(f"Human benchmark rates of serious-injury-or-worse and any-injury crashes are ~{ratios['Serious injury or worse']:.0f}x and "
      f"~{ratios['Any injury reported']:.0f}x the fatality rate used here, which is why statistically clear results")
print("exist for injury outcomes (see the CI table) long before they can exist for fatalities.")

The same logic applies to small fleets. Aurora reported 250,000 driverless miles without a collision attributed to its system (January 2026):

In [ ]:
k, miles = 0, 250_000
upper = stats.chi2.ppf(0.95, 2 * (k + 1)) / 2 / miles
print(f"Aurora: {k} collisions attributed to the Aurora Driver in {miles:,} driverless miles (Jan 2026).")
print(f"One-sided 95 % upper bound on that rate: {upper * 1e6:.1f} per million miles "
      f"(= {upper * 1e8:,.0f} per 100 million miles).")
print("Zero events in a small exposure supports only a weak bound - a reminder that 'no collisions so far' is")
print("an encouraging leading signal, not a demonstration of superhuman safety.")

**Interpretation.** Strong evidence on injury outcomes now exists for at least one operator, while fatality-level claims remain statistically weak for everyone. This is why modern frameworks (UL 4600, ISO/TS 5083:2025, the 2026 UN ADS Regulation) combine field statistics with a structured *safety case* and leading safety performance indicators rather than relying on miles alone.

---
## 5. Remote assistance is not remote driving

SAE J3016:2021 distinguishes:

- **Remote assistance** — a remote human provides information or advice to the ADS (e.g. "the lane is closed, the shoulder is passable"), while the **ADS remains responsible for the DDT**, including its own fallback.
- **Remote driving** — a remote driver performs (part of) the DDT in real time. This makes network latency, video quality and the operator's situational awareness part of the driving loop.

Waymo describes its *fleet response* as remote assistance: agents answer questions from the Waymo Driver, which "is in control of the vehicle at all times" ([Waymo, May 2024](https://waymo.com/blog/2024/05/fleet-response/)). In 2023 Cruise confirmed that its vehicles requested remote assistance frequently, as reported by [CNBC (6 Nov 2023)](https://www.cnbc.com/2023/11/06/cruise-confirms-robotaxis-rely-on-human-assistance-every-4-to-5-miles.html). German law formalises the role as *technical supervision* (Technische Aufsicht) for Level 4 operation (Act on Autonomous Driving, 2021; AFGBV, 2022).

**Safety implications.** (1) The ADS must reach a minimal risk condition on its own if the link or the answer does not arrive. (2) A wrong answer from a remote assistant is a new hazard source that belongs in the HARA/SOTIF analysis. (3) Staffing is a safety parameter: a request that waits is often a vehicle stopped in traffic.

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements or company data.

In [ ]:
# Remote assistance staffing as a queue (Erlang C). Illustrative parameters, NOT company data.
from math import factorial

def erlang_c_wait_prob(servers, offered_load):
    """Probability that a request has to wait in an M/M/c queue (Erlang C)."""
    a, c = offered_load, servers
    if a >= c:
        return 1.0
    s = sum(a ** k / factorial(k) for k in range(c))
    top = a ** c / factorial(c) * c / (c - a)
    return top / (s + top)

def p_wait_longer(t, servers, lam, mean_handle):
    """P(wait > t) in an M/M/c queue."""
    a = lam * mean_handle
    pw = erlang_c_wait_prob(servers, a)
    return pw * np.exp(-(servers - a) * t / mean_handle)

FLEET = 300                  # vehicles in service
REQ_PER_VEH_H = 1.0          # assistance requests per vehicle-hour (illustrative)
HANDLE_S = 60.0              # mean handling time per request [s] (illustrative)
lam = FLEET * REQ_PER_VEH_H / 3600
rows = []
for agents in (6, 8, 10, 15, 20, 30):
    if lam * HANDLE_S >= agents:
        rows.append({"agents": agents, "vehicles per agent": FLEET / agents, "P(wait > 0)": 1.0, "P(wait > 30 s)": 1.0,
                     "note": "unstable: queue grows without bound"})
        continue
    rows.append({"agents": agents, "vehicles per agent": round(FLEET / agents, 1),
                 "P(wait > 0)": round(erlang_c_wait_prob(agents, lam * HANDLE_S), 3),
                 "P(wait > 30 s)": round(p_wait_longer(30, agents, lam, HANDLE_S), 4), "note": ""})
ra = pd.DataFrame(rows)
display(ra)
ok = ra.loc[ra["P(wait > 30 s)"] <= 0.01]
print(f"Offered load: {lam * HANDLE_S:.1f} agent-equivalents. With these assumptions, "
      + (f"at least {int(ok['agents'].min())} agents ({ok['vehicles per agent'].max()} vehicles per agent) keep "
         "P(wait > 30 s) at or below 1 %." if len(ok) else "no staffing level in the table meets 1 %."))
print("Every waiting request is a vehicle that may be stopped in traffic: staffing is a safety parameter,")
print("and the ADS must reach an MRC on its own if no answer (or no connection) arrives.")

---
## 6. Incident reporting: the NHTSA Standing General Order

Since 2021, NHTSA's **Standing General Order (SGO) 2021-01** requires manufacturers and operators of vehicles equipped with ADS or Level 2 ADAS to report crashes. The **third amended SGO** was issued on 24 April 2025 and took effect on **16 June 2025** (as of October 2026; check NHTSA for later amendments):

- crashes involving a fatality, a person transported to hospital, a vulnerable road user, an airbag deployment or a tow-away: initial report within **five calendar days** (previously one day for the most serious categories);
- less severe ADS crashes (e.g. property damage expected above USD 1,000): **monthly** reports by the 15th of the following month;
- no monthly report is needed for months without reportable crashes, and only one reporting entity has to report a crash unless others hold materially different information.

Sources: [NHTSA SGO](https://www.nhtsa.gov/laws-regulations/standing-general-order-crash-reporting); [Sidley summary, 29 Apr 2025](https://environmentalhealthsafetybrief.sidley.com/2025/04/29/nhtsa-announces-new-policies-to-promote-autonomous-vehicles/).

**What SGO data can and cannot show.** The public data contain crash reports, not exposure: without miles driven per company, city and road type they cannot be turned into rates. Reports do not assign fault, can be incomplete or duplicated, and reporting thresholds differ from police-reported human crash data. Any comparison needs a matched benchmark — the hard part of Section 4. The SGO also does not require explanations of ADS decisions; it is a reporting duty. In Europe, ALKS under UN R157 must record occurrences in a *Data Storage System for Automated Driving* (DSSAD, paragraph 8).

### 6.1 Case: Cruise (2023 incident → 2024 shutdown)

On 2 October 2023 in San Francisco, a pedestrian was struck by a human-driven car and thrown into the path of a Cruise robotaxi; the robotaxi braked, then attempted to pull over and dragged her. On 24 October 2023 the California DMV suspended Cruise's driverless permits; Cruise then paused its driverless operations nationwide. In September 2024 NHTSA concluded a [consent order with Cruise concerning SGO crash reporting](https://www.nhtsa.gov/sites/nhtsa.gov/files/2024-09/cruise-consent-order-2024-web.pdf), and on 10 December 2024 GM announced that it would stop funding the robotaxi business. Lessons for safety engineering:

1. The **post-crash manoeuvre** — the fallback — caused additional harm. Fallback behaviour must be analysed in the same depth as nominal driving.
2. **Incomplete disclosure** to regulators destroyed trust faster than the incident itself. Reporting duties are part of the safety culture, and the safety case depends on them.
3. One incident ended a programme of more than USD 10 billion of investment (Fortune, Dec 2024): safety is a precondition for the business, not a cost centre.

The Uber ATG crash (2018) is analysed in detail in [Module 02](../02_Failures_and_Robustness/01_incident_case_studies.ipynb).

---
## 7. Open challenges (qualitative)

- **Scaling the ODD**: new cities, weather (snow), freeways and mixed traffic each require new evidence; ODD expansion is a safety-case change.
- **Statistical evidence for rare outcomes**: fatality-level claims need exposure that no fleet yet has; leading SPIs and structured arguments fill the gap (see [notebook 01](01_odd_and_runtime_monitoring.ipynb)).
- **Remote operations** at scale: staffing ratios, link loss, human factors of assistants.
- **Transparency and comparability**: exposure data and matched benchmarks are published by few operators.
- **Liability**: the revised EU Product Liability Directive (EU) 2024/2853 explicitly covers software; see Module 08 for the regulatory side.

---
## Key takeaways

1. As of October 2026, driverless robotaxis operate commercially in limited ODDs (Waymo in 14 cities; Baidu Apollo Go in 28 cities worldwide), driverless trucks run on defined lanes (Aurora), and consumer L3 is approved in narrow ODDs (Germany; pilots in China). Cruise was shut down (Dec 2024).
2. Deployment models differ in who drives, who performs the fallback and who is liable; supervised L2 is not automated driving.
3. Published RO data show large, statistically clear reductions in injury crashes for Waymo; fatality-level claims still need far more exposure. Always distinguish fatalities from injury crashes.
4. Zero events in a small exposure gives only a weak upper bound.
5. Remote assistance advises an ADS that stays responsible; remote driving puts the human and the network into the control loop. Staffing and link loss are safety parameters.
6. NHTSA's SGO (third amendment, effective 16 June 2025) mandates crash reporting but provides no exposure; rates require matched benchmarks.

## Self-assessment

1. Classify Waymo, Aurora, Mercedes DRIVE PILOT and Tesla FSD (Supervised) by deployment model and fallback.
2. Why are Waymo's injury-crash reductions statistically clearer than any fatality comparison?
3. Using the zero-failure formula, how many miles without a fatality are needed to show, with 95 % confidence, a rate below 1.09 per 100 million miles?
4. What does the 95 % upper bound for "zero collisions in 250,000 miles" tell you, and what does it not?
5. Distinguish remote assistance from remote driving. Which one adds network latency to the driving loop?
6. What does the third amended SGO require, and why can SGO counts not be compared directly between companies?

<details>
<summary>Answers</summary>

1. Waymo: L4 robotaxi, ADS performs the fallback to MRC. Aurora: L4 driverless trucking, ADS fallback. DRIVE PILOT: L3, transition demand to the fallback-ready user, MRM if no response. Tesla FSD (Supervised): L2, the driver supervises and is the fallback at all times.
2. Injury crashes are one to three orders of magnitude more frequent than fatal crashes, so hundreds of millions of miles already yield enough events for narrow confidence intervals; fatalities would need billions of miles for a 20 % difference.
3. n = −ln(0.05) / 1.09e−8 ≈ 275 million miles.
4. It bounds the rate at about 12 per million miles — far above human fatality or injury rates. It shows the absence of evidence of frequent collisions, not evidence of better-than-human safety.
5. Remote assistance: a human gives information to an ADS that keeps driving. Remote driving: the human performs the DDT remotely, so network latency and reliability enter the control loop.
6. Reports within five calendar days for crashes with fatality, hospital transport, VRU involvement, airbag deployment or tow-away; monthly reports for less severe ADS crashes. Counts lack exposure (miles, roads, cities) and fault information, so they are not rates.
</details>

## References

- Waymo (2026). *Safety Impact* (data through June 2026). https://waymo.com/safety/impact/
- Kusano, K. D., Scanlon, J. M., Chen, Y.-H., McMurry, T. L., Chen, R., Gode, T., Victor, T. (2024). Comparison of Waymo rider-only crash data to human benchmarks at 7.1 million miles. *Traffic Injury Prevention*, 25(sup1). https://doi.org/10.1080/15389588.2024.2380786
- Kalra, N., Paddock, S. M. (2016). Driving to safety: How many miles of driving would it take to demonstrate autonomous vehicle reliability? *Transportation Research Part A*, 94, 182–193. https://doi.org/10.1016/j.tra.2016.09.010
- NHTSA. *Standing General Order on Crash Reporting* (SGO 2021-01, third amendment effective 16 June 2025). https://www.nhtsa.gov/laws-regulations/standing-general-order-crash-reporting
- NTSB (2019). *Collision Between Vehicle Controlled by Developmental Automated Driving System and Pedestrian, Tempe, Arizona, March 18, 2018*. HAR-19/03. https://www.ntsb.gov/investigations/Pages/HWY18MH010.aspx
- UN Regulation No. 157 (ALKS), incl. 01 series of amendments; DSSAD requirements in paragraph 8.
- UNECE / WP.29 (June 2026). Adoption of a UN Regulation and a Global Technical Regulation on Automated Driving Systems. Summary: https://www.connectedautomateddriving.eu/blog/unece-adopts-first-ever-global-rules-allowing-fully-autonomous-vehicles/
- SAE J3016 (2021). *Taxonomy and Definitions for Terms Related to Driving Automation Systems for On-Road Motor Vehicles*.
- UL 4600 (3rd ed., 2023); ISO/TS 5083:2025.
- Germany: Act on Autonomous Driving (2021), StVG §§ 1d–1l; Autonomous Vehicles Approval and Operation Ordinance (AFGBV, 2022).
- Directive (EU) 2024/2853 on liability for defective products.
- Company and press sources for Section 3 are linked directly below the landscape table.

---
**Navigation:** ← [V2X and cooperative driving](02_v2x_cooperative_driving.ipynb) · [Module index](README.md) · [Module 08: Standards, Regulation and Safety Case](../08_Standards_Regulation_and_Safety_Case/README.md) →